# Missionários em segurança com canibais
O intuito desse trabalho é gerenciar o uso do barco para que todos os missionários da paróquia do Frei Sardinha consiga atravessar o rio em segurança na companhia dos queridos indígenas Caetés. Mas para garantir a segurança de todos, principalmente dos missionários, existem algumas regras que devem ser respeitadas.

Primeiramente, o problema considera duas margens do rio: a margem inicial (esquerda) e a margem de destino (direita). Todos os indivíduos começam na margem inicial e devem ser transportados para a margem de destino.

O transporte é realizado por meio de um barco, que possui capacidade limitada, podendo transportar no máximo um número fixo de pessoas por travessia.

## Regras

As regras que regem o problema são as seguintes:

- O barco deve sempre transportar pelo menos uma pessoa, o barco nunca se desloca sozinho.
- Em nenhuma das margens do rio pode haver um número de indígenas maior que o de missionários, caso exista pelo menos um missionário presente naquela margem. Caso contrário, os missionários seriam colocados em risco.

# Índice

1. [Como usar](#Como-usar)
2. [Definições iniciais](#Definições-iniciais)
3. [Validação](#Validação)
4. [BFS - Busca em Largura](#BFS---Busca-em-Largura)
5. [DFS - Busca em Profundidade](#DFS---Busca-em-Profundidade)
6. [DLS - Busca em Profundidade Limitada](#DLS---Busca-em-Profundidade-Limitada)
7. [Playground](#Playground)

## Como usar
com todas as células compiladas é possível utilizar no final do arquivo o playground, todos os 3 cenários estão prontos com métricas para serem retornados, é possível configurar o número de missionários, de canibais e o tamanho do barco em todos os algorítmos, no de Busca por Profundidade limitada também é possível definir, o limite inicial e o limite máximo.

## Definições iniciais

In [53]:
from time import perf_counter
from collections import deque

# possibilita que os parâmetros possam ser redefinidos conforme necessidade do cenário
def set_initial_state(missionaries=3, cannibals=3, boat="left"):
    state = {
        "left": (missionaries, cannibals),
        "right": (0, 0),
        "boat": boat
    }
    return state
initial_state = set_initial_state()

#melhora a qualidade de exibição das métricas e estatísticas
def print_metrics(metrics):
    match(metrics):
        case {"tipo": "dls", "sucesso": False}:
            print(f"❌ Tempo de execução: {metrics['tempo_de_execução']:.6f} s")
        case _:
            print("\n📊 MÉTRICAS DE EXECUÇÃO")
            print("=" * 35)
        
            print(f"Sucesso: {'✅' if metrics['sucesso'] else '❌'}")
        
            print(f"Tempo de execução: {metrics['tempo_de_execução']:.6f} s")
        
            print(f"Estados verificados: {metrics['estados_verificados']:,}".replace(",", "."))
            print(f"Nós gerados: {metrics['nodes_gerados']:,}".replace(",", "."))
            print(f"Nós expandidos: {metrics['nodes_expandidos']:,}".replace(",", "."))
        
            print(f"Tamanho máximo da {metrics['estrutura']}: {metrics['tamanho_maximo_estrutura']:,}".replace(",", "."))
        
            profundidade = metrics["profundidade_solucao"]
            if profundidade is not None:
                print(f"Profundidade da solução: {profundidade}")
            else:
                print("Profundidade da solução: —")
        
            print("=" * 35)

### Validação
A validação consiste em verificar em cada margem, se há mais canibais que missionarios.

In [2]:
def is_valid(state):
    left_missionaries, left_cannibals = state["left"]
    right_missionaries, right_cannibals = state["right"]

    if min(left_missionaries, left_cannibals, right_missionaries, right_cannibals) < 0:
        return False

    if left_missionaries > 0 and left_cannibals > left_missionaries:
        return False

    if right_missionaries > 0 and right_cannibals > right_missionaries:
        return False

    return True

### Função sucessor
É listado todos os movimentos possíveis de acordo com a quantidade de lugares no barco, por meio da soma do valor dos index, é possível comparar quantas pessoas de cada cultura cabe no espaço do barco.

In [3]:
def possible_moves(boat_capacity):
    moves = []
    
    for m in range(boat_capacity + 1):
        for c in range(boat_capacity + 1):
            total = m + c
            
            if 1 <= total <= boat_capacity:
                moves.append((m, c))
    
    return moves

In [4]:
possible_moves(2) # Um exemplo de uso com o máximo de duas pessoas no barco

[(0, 1), (0, 2), (1, 0), (1, 1), (2, 0)]

Com isso, podemos finalmente criar a função que aplica os movimentos:

In [5]:
def apply_move(state, move):
    missionaries, cannibals = move
    left_missionaries, left_cannibals = state["left"]
    right_missionaries, right_cannibals = state["right"]

    if state["boat"] == "left":
        return {
            "left": (left_missionaries - missionaries, left_cannibals - cannibals),
            "right": (right_missionaries + missionaries, right_cannibals + cannibals),
            "boat": "right"
        }
    else:
        return {
            "left": (left_missionaries + missionaries, left_cannibals + cannibals),
            "right": (right_missionaries - missionaries, right_cannibals - cannibals),
            "boat": "left"
        }

In [6]:
# Test
initial_state = set_initial_state(3, 3)
new_state = apply_move(initial_state, (2, 1))
print(new_state)
print("Válido?", is_valid(new_state))

{'left': (1, 2), 'right': (2, 1), 'boat': 'right'}
Válido? False


### Explicação da função sucessor
A função sucessor é responsável por gerar, a partir de um estado atual, todos os próximos estados possíveis que podem ser alcançados em **uma única travessia** do barco. Para isso, ela combina três etapas já definidas anteriormente:

1. Obtém todos os movimentos possíveis por meio da função `possible_moves()` (combinações de missionários e canibais que cabem no barco).
2. Aplica cada movimento ao estado atual com a função `apply_move(state, move)`, produzindo um novo estado candidato.
3. Valida esse novo estado com `is_valid(new_state)`, descartando automaticamente estados que violam as regras do problema.

Ao final, a função retorna uma lista de pares `(movimento, novo_estado)`. Essa estrutura é essencial para os algoritmos de busca (como o BFS), pois permite explorar o espaço de estados de forma sistemática, mantendo apenas transições válidas e seguras.

In [7]:
def successors(state, boat_capacity):
    result = []

    for move in possible_moves(boat_capacity):
        new_state = apply_move(state, move)

        if is_valid(new_state):
            result.append((move, new_state))

    return result

In [8]:
def is_goal(state):
    return state["left"] == (0, 0)

## BFS - Busca em Largura
Com o auxílo de uma fila, o algorítmo percorre linha por linha da arvore, mantendo o mesmo nível na busca por uma solução, tende a percorrer a árvore toda e só para quando encontra um resultado, o resultado tende a ser o mais curto em passos.

In [116]:
from collections import deque
from time import perf_counter

def solve_bfs(initial_state, boat_capacity, optimized=False):
    start_time = perf_counter()

    def state_to_key(state):
        return (state["left"], state["right"], state["boat"])

    initial_key = state_to_key(initial_state)

    queue = deque([{
        "state": initial_state,
        "path": [],
        "movements": [(0, 0)],
        "path_keys": {initial_key}
    }])

    visited = {initial_key} if optimized else None

    generated_nodes = 1
    expanded_nodes = 0
    max_queue_size = 1

    while queue:
        max_queue_size = max(max_queue_size, len(queue))

        node = queue.popleft()
        expanded_nodes += 1

        state = node["state"]
        path = node["path"]
        movements = node["movements"]
        path_keys = node["path_keys"]

        if is_goal(state):
            end_time = perf_counter()
            solution = path + [state]

            success_metrics = {
                "sucesso": True,
                "tempo_de_execução": end_time - start_time,
                "estados_verificados": len(visited) if optimized else generated_nodes,
                "nodes_gerados": generated_nodes,
                "nodes_expandidos": expanded_nodes,
                "tamanho_maximo_estrutura": max_queue_size,
                "estrutura": "fila",
                "profundidade_solucao": len(solution) - 1,
                "tipo": "bfs",
                "otimizado": optimized
            }

            return solution, movements, success_metrics

        for move, new_state in successors(state, boat_capacity):
            state_key = state_to_key(new_state)

            if optimized:
                # Evita revisitar estados já descobertos globalmente
                if state_key in visited:
                    continue
            else:
                # Evita apenas ciclos dentro do caminho atual
                if state_key in path_keys:
                    continue

            queue.append({
                "state": new_state,
                "path": path + [state],
                "movements": movements + [move],
                "path_keys": path_keys | {state_key}
            })

            if optimized:
                visited.add(state_key)

            generated_nodes += 1

    end_time = perf_counter()

    failed_metrics = {
        "sucesso": False,
        "tempo_de_execução": end_time - start_time,
        "estados_verificados": len(visited) if optimized else generated_nodes,
        "nodes_gerados": generated_nodes,
        "nodes_expandidos": expanded_nodes,
        "tamanho_maximo_estrutura": max_queue_size,
        "estrutura": "fila",
        "profundidade_solucao": None,
        "tipo": "bfs",
        "otimizado": optimized
    }

    return None, None, failed_metrics

In [117]:
def scenario_bfs(missionaries = 3, cannibals = 3, boat_size = 2, optimized = True):
    initial_state = set_initial_state(missionaries, cannibals)
    solution, movements, metrics = solve_bfs(initial_state, boat_size, optimized)
    
    if solution and movements:
        
        total_steps = len(movements) - 1  # ignorando (0,0)
        print(f"Resolvido em {total_steps} passos.")
    
        if total_steps > 20:
            print("Solução muito longa, não será exibida.")
        else:
            print("\nGabarito da travessia:")
            boat_side = "left"
        
            for step, move in enumerate(movements[1:], start=1):
                missionaries, cannibals = move
                origin = "esquerda" if boat_side == "left" else "direita"
                destination = "direita" if boat_side == "left" else "esquerda"
        
                missionaries_label = "missionário" if missionaries == 1 else "missionários"
                cannibals_label = "canibal" if cannibals == 1 else "canibais"
        
                print(
                    f"Passo {step}: levar {missionaries} {missionaries_label} e "
                    f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
                )
        
                boat_side = "right" if boat_side == "left" else "left"
    else:
        print("Nenhuma solução encontrada")
    
    print_metrics(metrics)
scenario_bfs()

Resolvido em 11 passos.

Gabarito da travessia:
Passo 1: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 2: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 3: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 4: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 5: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 6: levar 1 missionário e 1 canibal da margem direita para a margem esquerda.
Passo 7: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 8: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 9: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 10: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 11: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.

📊 MÉTRICAS DE EXECUÇ

------------

## DFS - Busca em Profundidade
A busca em profundidade explora um caminho até o fim antes de voltar (backtracking) para tentar alternativas. Aqui, usamos uma pilha para controlar os estados pendentes e também evitamos revisitar estados já explorados.

In [118]:
from time import perf_counter

def solve_dfs(initial_state, boat_capacity, optimized=False):
    start_time = perf_counter()

    def state_to_key(state):
        return (state["left"], state["right"], state["boat"])

    initial_key = state_to_key(initial_state)

    stack = [{
        "state": initial_state,
        "path": [],
        "movements": [(0, 0)],
        "path_keys": {initial_key}
    }]

    visited = {initial_key} if optimized else None

    generated_nodes = 1
    expanded_nodes = 0
    max_stack_size = 1

    while stack:
        max_stack_size = max(max_stack_size, len(stack))

        node = stack.pop()
        expanded_nodes += 1

        state = node["state"]
        path = node["path"]
        movements = node["movements"]
        path_keys = node["path_keys"]

        if is_goal(state):
            end_time = perf_counter()
            solution = path + [state]

            success_metrics = {
                "sucesso": True,
                "tempo_de_execução": end_time - start_time,
                "estados_verificados": len(visited) if optimized else generated_nodes,
                "nodes_gerados": generated_nodes,
                "nodes_expandidos": expanded_nodes,
                "tamanho_maximo_estrutura": max_stack_size,
                "estrutura": "pilha",
                "profundidade_solucao": len(solution) - 1,
                "tipo": "dfs",
                "otimizado": optimized
            }

            return solution, movements, success_metrics

        for move, new_state in successors(state, boat_capacity):
            state_key = state_to_key(new_state)

            if optimized:
                # Evita revisitar estados já explorados globalmente
                if state_key in visited:
                    continue
            else:
                # Evita ciclos apenas no caminho atual
                if state_key in path_keys:
                    continue

            stack.append({
                "state": new_state,
                "path": path + [state],
                "movements": movements + [move],
                "path_keys": path_keys | {state_key}
            })

            if optimized:
                visited.add(state_key)

            generated_nodes += 1

    end_time = perf_counter()

    failed_metrics = {
        "sucesso": False,
        "tempo_de_execução": end_time - start_time,
        "estados_verificados": len(visited) if optimized else generated_nodes,
        "nodes_gerados": generated_nodes,
        "nodes_expandidos": expanded_nodes,
        "tamanho_maximo_estrutura": max_stack_size,
        "estrutura": "pilha",
        "profundidade_solucao": None,
        "tipo": "dfs",
        "otimizado": optimized
    }

    return None, None, failed_metrics

In [119]:
def scenario_dfs(missionaries = 3, cannibals = 3, boat_size = 2, optimized = True):
    initial_state = set_initial_state(missionaries, cannibals)
    solution_dfs, movements_dfs, metrics = solve_dfs(initial_state, boat_size, optimized)
    
    if solution_dfs and movements_dfs:
        
        total_steps = len(movements_dfs) - 1  # ignorando (0,0)
        print(f"Resolvido em {total_steps} passos.")
    
        if total_steps > 20:
            print("Solução muito longa, não será exibida.")
        else:
            print("\nGabarito da travessia (DFS):")
            boat_side = "left"
        
            for step, move in enumerate(movements_dfs[1:], start=1):
                missionaries, cannibals = move
                origin = "esquerda" if boat_side == "left" else "direita"
                destination = "direita" if boat_side == "left" else "esquerda"
        
                missionaries_label = "missionário" if missionaries == 1 else "missionários"
                cannibals_label = "canibal" if cannibals == 1 else "canibais"
        
                print(
                    f"Passo {step}: levar {missionaries} {missionaries_label} e "
                    f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
                )
        
                boat_side = "right" if boat_side == "left" else "left"
    else:
        print("Nenhuma solução encontrada com DFS")
    
    print_metrics(metrics)
scenario_dfs()

Resolvido em 11 passos.

Gabarito da travessia (DFS):
Passo 1: levar 1 missionário e 1 canibal da margem esquerda para a margem direita.
Passo 2: levar 1 missionário e 0 canibais da margem direita para a margem esquerda.
Passo 3: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 4: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 5: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 6: levar 1 missionário e 1 canibal da margem direita para a margem esquerda.
Passo 7: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 8: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 9: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 10: levar 1 missionário e 0 canibais da margem direita para a margem esquerda.
Passo 11: levar 1 missionário e 1 canibal da margem esquerda para a margem direita.

📊 MÉTRICAS DE EXEC

------------

## DLS - Busca em Profundidade Limitada
A busca em profundidade limitada (Depth-Limited Search) funciona como a DFS, mas com um limite máximo de profundidade. Isso evita que a busca avance indefinidamente em ramos muito longos e permite controlar o custo de exploração.

In [106]:
def solve_dls(initial_state, current_depth_limit, boat_capacity, optimized=False):
    start_time = perf_counter()

    def state_to_key(state):
        return (state["left"], state["right"], state["boat"])

    initial_key = state_to_key(initial_state)

    stack = [{
        "state": initial_state,
        "path": [],
        "movements": [(0, 0)],
        "depth": 0,
        "path_keys": {initial_key}
    }]

    visited = {initial_key} if optimized else None

    generated_nodes = 1
    expanded_nodes = 0
    max_stack_size = 1

    while stack:
        max_stack_size = max(max_stack_size, len(stack))

        node = stack.pop()
        expanded_nodes += 1

        state = node["state"]
        path = node["path"]
        movements = node["movements"]
        depth = node["depth"]
        path_keys = node["path_keys"]

        if is_goal(state):
            end_time = perf_counter()
            solution = path + [state]

            success_metrics = {
                "sucesso": True,
                "tempo_de_execução": end_time - start_time,
                "estados_verificados": len(visited) if optimized else generated_nodes,
                "nodes_gerados": generated_nodes,
                "nodes_expandidos": expanded_nodes,
                "tamanho_maximo_estrutura": max_stack_size,
                "estrutura": "pilha",
                "profundidade_solucao": len(solution) - 1,
                "tipo": "dls",
                "otimizado": optimized
            }

            return solution, movements, success_metrics

        if depth >= current_depth_limit:
            continue

        for move, new_state in successors(state, boat_capacity):
            state_key = state_to_key(new_state)

            if optimized:
                # Evita revisitar estados já explorados globalmente
                if state_key in visited:
                    continue
            else:
                # Evita ciclos no caminho atual da busca em profundidade
                if state_key in path_keys:
                    continue

            stack.append({
                "state": new_state,
                "path": path + [state],
                "movements": movements + [move],
                "depth": depth + 1,
                "path_keys": path_keys | {state_key}
            })

            if optimized:
                visited.add(state_key)

            generated_nodes += 1

    end_time = perf_counter()

    failed_metrics = {
        "sucesso": False,
        "tempo_de_execução": end_time - start_time,
        "estados_verificados": len(visited) if optimized else generated_nodes,
        "nodes_gerados": generated_nodes,
        "nodes_expandidos": expanded_nodes,
        "tamanho_maximo_estrutura": max_stack_size,
        "estrutura": "pilha",
        "profundidade_solucao": None,
        "tipo": "dls",
        "otimizado": optimized
    }

    return None, None, failed_metrics

In [105]:
def scenario_dls(missionaries = 3, cannibals = 3, boat_size = 2, depth_limit = 0, max_depth_limit = 500, optimized=True):

    initial_state = set_initial_state(missionaries, cannibals)
    solution_dls, movements_dls = None, None
    
    while depth_limit <= max_depth_limit:
        solution_dls, movements_dls, metrics = solve_dls(initial_state, depth_limit, boat_size, optimized)
    
        if solution_dls is None:
            print(f"depth_limit = {depth_limit}: nenhuma solução encontrada.")
            print_metrics(metrics)
            depth_limit += 1
            continue
    
        print(f"depth_limit = {depth_limit}: solução encontrada!")
        break
    
    if solution_dls is None:
        print(
            f"\nNenhuma solução encontrada até depth_limit = {max_depth_limit}."
        )
    else:
        print("\nGabarito da travessia (DLS):")
        boat_side = "left"
    
        for step, move in enumerate(movements_dls[1:], start=1):
            missionaries, cannibals = move
            origin = "esquerda" if boat_side == "left" else "direita"
            destination = "direita" if boat_side == "left" else "esquerda"
    
            missionaries_label = "missionário" if missionaries == 1 else "missionários"
            cannibals_label = "canibal" if cannibals == 1 else "canibais"
    
            print(
                f"Passo {step}: levar {missionaries} {missionaries_label} e "
                f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
            )
    
            boat_side = "right" if boat_side == "left" else "left"
        print_metrics(metrics)
scenario_dls()

depth_limit = 0: nenhuma solução encontrada.
❌ Tempo de execução: 0.000009 s
depth_limit = 1: nenhuma solução encontrada.
❌ Tempo de execução: 0.000037 s
depth_limit = 2: nenhuma solução encontrada.
❌ Tempo de execução: 0.000042 s
depth_limit = 3: nenhuma solução encontrada.
❌ Tempo de execução: 0.000042 s
depth_limit = 4: nenhuma solução encontrada.
❌ Tempo de execução: 0.000047 s
depth_limit = 5: nenhuma solução encontrada.
❌ Tempo de execução: 0.000053 s
depth_limit = 6: nenhuma solução encontrada.
❌ Tempo de execução: 0.000058 s
depth_limit = 7: nenhuma solução encontrada.
❌ Tempo de execução: 0.000063 s
depth_limit = 8: nenhuma solução encontrada.
❌ Tempo de execução: 0.000072 s
depth_limit = 9: nenhuma solução encontrada.
❌ Tempo de execução: 0.000077 s
depth_limit = 10: nenhuma solução encontrada.
❌ Tempo de execução: 0.000093 s
depth_limit = 11: solução encontrada!

Gabarito da travessia (DLS):
Passo 1: levar 1 missionário e 1 canibal da margem esquerda para a margem direita.
P

## Playground

O playground deste notebook expõe três funções públicas para testar diferentes estratégias de busca no problema dos missionários e canibais.

### Funções públicas

#### `scenario_bfs(missionaries=3, cannibals=3, boat_size=2, optimized = True)`

Executa o problema usando **Busca em Largura (BFS)**.

Parâmetros:
- `missionaries`: quantidade inicial de missionários na margem esquerda
- `cannibals`: quantidade inicial de canibais na margem esquerda
- `boat_size`: capacidade máxima do barco
- `optimized`: capacidade de usar ou não um algorítmo otimizado que verifica se o estado já foi consultado em algum momento nessa profundidade, evitando retorno a estados já computados globalmente.

---

#### `scenario_dfs(missionaries=3, cannibals=3, boat_size=2, optimized = True)`

Executa o problema usando **Busca em Profundidade (DFS)**.

Parâmetros:
- `missionaries`: quantidade inicial de missionários na margem esquerda
- `cannibals`: quantidade inicial de canibais na margem esquerda
- `boat_size`: capacidade máxima do barco
- `optimized`: capacidade de usar ou não um algorítmo otimizado que verifica se o estado já foi consultado em algum momento nessa profundidade, evitando retorno a estados já computados globalmente.

---

#### `scenario_dls(missionaries=3, cannibals=3, boat_size=2, depth_limit=0, max_depth_limit=500, optimized = True)`

Executa o problema usando **Busca em Profundidade Limitada (DLS)**, tentando encontrar solução a partir de um limite inicial de profundidade.

Parâmetros:
- `missionaries`: quantidade inicial de missionários na margem esquerda
- `cannibals`: quantidade inicial de canibais na margem esquerda
- `boat_size`: capacidade máxima do barco
- `depth_limit`: limite inicial de profundidade
- `max_depth_limit`: limite máximo de profundidade a ser testado
- `optimized`: capacidade de usar ou não um algorítmo otimizado que verifica se o estado já foi consultado em algum momento nessa profundidade, evitando retorno a estados já computados globalmente.

### Exemplos de uso

```python
scenario_bfs()
scenario_dfs(optimized = False)

scenario_bfs(missionaries=4, cannibals=4)
scenario_dfs(4, 3, 3)
scenario_dfs(4, 3, 3, False)
scenario_dls(missionaries=3, cannibals=3, boat_size=2, depth_limit=0, max_depth_limit=20)

In [120]:
scenario_bfs(missionaries=41)

Resolvido em 85 passos.
Solução muito longa, não será exibida.

📊 MÉTRICAS DE EXECUÇÃO
Sucesso: ✅
Tempo de execução: 0.006246 s
Estados verificados: 318
Nós gerados: 318
Nós expandidos: 318
Tamanho máximo da fila: 5
Profundidade da solução: 85


In [121]:
scenario_dls(missionaries=5, cannibals=5, boat_size=3, depth_limit=0, max_depth_limit=20)

depth_limit = 0: nenhuma solução encontrada.
❌ Tempo de execução: 0.000011 s
depth_limit = 1: nenhuma solução encontrada.
❌ Tempo de execução: 0.000059 s
depth_limit = 2: nenhuma solução encontrada.
❌ Tempo de execução: 0.000092 s
depth_limit = 3: nenhuma solução encontrada.
❌ Tempo de execução: 0.000112 s
depth_limit = 4: nenhuma solução encontrada.
❌ Tempo de execução: 0.000135 s
depth_limit = 5: nenhuma solução encontrada.
❌ Tempo de execução: 0.000157 s
depth_limit = 6: nenhuma solução encontrada.
❌ Tempo de execução: 0.000158 s
depth_limit = 7: nenhuma solução encontrada.
❌ Tempo de execução: 0.000183 s
depth_limit = 8: nenhuma solução encontrada.
❌ Tempo de execução: 0.000194 s
depth_limit = 9: nenhuma solução encontrada.
❌ Tempo de execução: 0.000225 s
depth_limit = 10: nenhuma solução encontrada.
❌ Tempo de execução: 0.000222 s
depth_limit = 11: nenhuma solução encontrada.
❌ Tempo de execução: 0.000283 s
depth_limit = 12: nenhuma solução encontrada.
❌ Tempo de execução: 0.00030

In [122]:
scenario_bfs(missionaries=30, cannibals=10, boat_size=5)

Resolvido em 19 passos.

Gabarito da travessia:
Passo 1: levar 0 missionários e 4 canibais da margem esquerda para a margem direita.
Passo 2: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 3: levar 4 missionários e 1 canibal da margem esquerda para a margem direita.
Passo 4: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 5: levar 2 missionários e 3 canibais da margem esquerda para a margem direita.
Passo 6: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 7: levar 2 missionários e 3 canibais da margem esquerda para a margem direita.
Passo 8: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 9: levar 2 missionários e 3 canibais da margem esquerda para a margem direita.
Passo 10: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 11: levar 4 missionários e 1 canibal da margem esquerda para a margem direita.
Passo 12: levar 0 miss

In [123]:
scenario_dfs(missionaries=30, cannibals=10, boat_size=5)

Resolvido em 39 passos.
Solução muito longa, não será exibida.

📊 MÉTRICAS DE EXECUÇÃO
Sucesso: ✅
Tempo de execução: 0.004994 s
Estados verificados: 374
Nós gerados: 374
Nós expandidos: 134
Tamanho máximo da pilha: 241
Profundidade da solução: 39


50 missionários, 30 canibais, 4 posições no barco. Para a solução bfs que encontrou os 53 passos houve um tempo 

In [81]:
scenario_bfs(50, 30, 4)

Resolvido em 53 passos.
Solução muito longa, não será exibida.

📊 MÉTRICAS DE EXECUÇÃO
Sucesso: ✅
Tempo de execução: 0.030144 s
Estados verificados: 1.378
Nós gerados: 1.378
Nós expandidos: 1.375
Tamanho máximo da fila: 38
Profundidade da solução: 53


In [82]:
scenario_dfs(50, 30, 4)

Resolvido em 109 passos.
Solução muito longa, não será exibida.

📊 MÉTRICAS DE EXECUÇÃO
Sucesso: ✅
Tempo de execução: 0.042929 s
Estados verificados: 1.055
Nós gerados: 1.055
Nós expandidos: 445
Tamanho máximo da pilha: 611
Profundidade da solução: 109


In [102]:
scenario_dls(50, 30, 4, 110)

depth_limit = 110: solução encontrada!

Gabarito da travessia (DLS):
Passo 1: levar 4 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 2: levar 3 missionários e 0 canibais da margem direita para a margem esquerda.
Passo 3: levar 4 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 4: levar 1 missionário e 0 canibais da margem direita para a margem esquerda.
Passo 5: levar 4 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 6: levar 3 missionários e 0 canibais da margem direita para a margem esquerda.
Passo 7: levar 4 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 8: levar 1 missionário e 0 canibais da margem direita para a margem esquerda.
Passo 9: levar 4 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 10: levar 3 missionários e 0 canibais da margem direita para a margem esquerda.
Passo 11: levar 4 missionários e 0 canibais da margem esquerda para a margem direi

Abaixo é um cenário antes da inserção da otimização, considere como optimized false, mesmo não tendo esse parâmetro. Não removi devido à sua longa duração

In [75]:
scenario_dls(missionaries=30, cannibals=10, boat_size=5, depth_limit=0, max_depth_limit=30)

depth_limit = 0: nenhuma solução encontrada.
❌ Tempo de execução: 0.000010 s
depth_limit = 1: nenhuma solução encontrada.
❌ Tempo de execução: 0.000093 s
depth_limit = 2: nenhuma solução encontrada.
❌ Tempo de execução: 0.000810 s
depth_limit = 3: nenhuma solução encontrada.
❌ Tempo de execução: 0.005949 s
depth_limit = 4: nenhuma solução encontrada.
❌ Tempo de execução: 0.016745 s
depth_limit = 5: nenhuma solução encontrada.
❌ Tempo de execução: 0.109610 s
depth_limit = 6: nenhuma solução encontrada.
❌ Tempo de execução: 1.255668 s
depth_limit = 7: nenhuma solução encontrada.
❌ Tempo de execução: 13.024629 s
depth_limit = 8: nenhuma solução encontrada.
❌ Tempo de execução: 159.790013 s
depth_limit = 9: nenhuma solução encontrada.
❌ Tempo de execução: 1981.858470 s
depth_limit = 10: nenhuma solução encontrada.
❌ Tempo de execução: 25745.846772 s


KeyboardInterrupt: 

E comparando com sua versão otimizada, é gritante a diferença.

In [103]:
scenario_dls(50, 30, 4)

depth_limit = 0: nenhuma solução encontrada.
❌ Tempo de execução: 0.000010 s
depth_limit = 1: nenhuma solução encontrada.
❌ Tempo de execução: 0.000094 s
depth_limit = 2: nenhuma solução encontrada.
❌ Tempo de execução: 0.000391 s
depth_limit = 3: nenhuma solução encontrada.
❌ Tempo de execução: 0.000548 s
depth_limit = 4: nenhuma solução encontrada.
❌ Tempo de execução: 0.000681 s
depth_limit = 5: nenhuma solução encontrada.
❌ Tempo de execução: 0.000745 s
depth_limit = 6: nenhuma solução encontrada.
❌ Tempo de execução: 0.000759 s
depth_limit = 7: nenhuma solução encontrada.
❌ Tempo de execução: 0.001288 s
depth_limit = 8: nenhuma solução encontrada.
❌ Tempo de execução: 0.000905 s
depth_limit = 9: nenhuma solução encontrada.
❌ Tempo de execução: 0.000823 s
depth_limit = 10: nenhuma solução encontrada.
❌ Tempo de execução: 0.000816 s
depth_limit = 11: nenhuma solução encontrada.
❌ Tempo de execução: 0.000994 s
depth_limit = 12: nenhuma solução encontrada.
❌ Tempo de execução: 0.00082

Uma lógica para o algorítmo DLS é iniciar no nó que contém o resultado, seria então garantido ter uma resposta e talvez fosse até rápido, porém, se o tempo no limite 10 foi de 25745s(aproximadamente 7 horas), mesmo estando na profundidade de nó correto, ele teria que testar vários caminhos de 109 nós pra algum deles ser o correto, o que ainda pode gerar um tempo muito grande no algorítmo não otimizado.